# Module 04: ZeRO Optimization (FSDP + DeepSpeed)

Read `README.md` before starting.

**Goal**: Shard model state across GPUs to fit larger models and reduce per-GPU memory.

In [ ]:
!pip install -q deepspeed==0.14.0

import torch
assert torch.cuda.device_count() == 2
print(f"Ready. GPUs: {torch.cuda.device_count()}")

## Part 1: FSDP — Three Sharding Strategies Side by Side

We run the same model under `NO_SHARD` (DDP), `SHARD_GRAD_OP` (ZeRO-2), and `FULL_SHARD` (ZeRO-3).
Compare the memory numbers.

In [ ]:
%%writefile /kaggle/working/fsdp_compare.py
"""
Compares FSDP sharding strategies side by side.
Run: torchrun --nproc_per_node=2 fsdp_compare.py --strategy FULL_SHARD
"""
import os
import argparse
import functools
import time

import torch
import torch.nn as nn
import torch.distributed as dist
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP, ShardingStrategy
from torch.distributed.fsdp.fully_sharded_data_parallel import CPUOffload
from torch.distributed.fsdp.wrap import transformer_auto_wrap_policy
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.data.distributed import DistributedSampler

# ─── Config ───────────────────────────────────────────────────────────────────
VOCAB_SIZE = 50000
D_MODEL = 1024
NHEAD = 16
NUM_LAYERS = 12
BATCH_SIZE = 4
SEQ_LEN = 128
DATASET_SIZE = 200
NUM_STEPS = 10

STRATEGY_MAP = {
    "NO_SHARD":      ShardingStrategy.NO_SHARD,        # DDP
    "SHARD_GRAD_OP": ShardingStrategy.SHARD_GRAD_OP,   # ZeRO-2
    "FULL_SHARD":    ShardingStrategy.FULL_SHARD,       # ZeRO-3
}

# ─── Model ────────────────────────────────────────────────────────────────────
class TransformerLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(VOCAB_SIZE, D_MODEL)
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(
                D_MODEL, NHEAD, D_MODEL * 4,
                dropout=0.1, batch_first=True
            )
            for _ in range(NUM_LAYERS)
        ])
        self.head = nn.Linear(D_MODEL, VOCAB_SIZE)

    def forward(self, x):
        x = self.embedding(x)
        for layer in self.layers:
            x = layer(x)
        return self.head(x)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--strategy", default="FULL_SHARD",
                        choices=["NO_SHARD", "SHARD_GRAD_OP", "FULL_SHARD"])
    parser.add_argument("--cpu_offload", action="store_true")
    args = parser.parse_args()

    rank = int(os.environ["RANK"])
    local_rank = int(os.environ["LOCAL_RANK"])
    world_size = int(os.environ["WORLD_SIZE"])
    dist.init_process_group(backend="nccl")
    torch.cuda.set_device(local_rank)

    if rank == 0:
        print(f"Strategy: {args.strategy} | CPU offload: {args.cpu_offload}")

    # Build model
    model = TransformerLM()
    total_params = sum(p.numel() for p in model.parameters())
    if rank == 0:
        print(f"Parameters: {total_params:,} (~{total_params * 16 / 1e9:.1f} GB full training)")

    # FSDP wrap policy: wrap each TransformerEncoderLayer independently
    # This means each layer's params are a separate shard unit
    wrap_policy = functools.partial(
        transformer_auto_wrap_policy,
        transformer_layer_cls={nn.TransformerEncoderLayer}
    )

    model = FSDP(
        model,
        auto_wrap_policy=wrap_policy,
        sharding_strategy=STRATEGY_MAP[args.strategy],
        cpu_offload=CPUOffload(offload_params=args.cpu_offload),
        device_id=local_rank,
        # Mixed precision: compute in fp16, accumulate in fp32
        mixed_precision=None,  # Set to MixedPrecision() for fp16 training
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

    dataset = TensorDataset(
        torch.randint(0, VOCAB_SIZE, (DATASET_SIZE, SEQ_LEN)),
        torch.randint(0, VOCAB_SIZE, (DATASET_SIZE, SEQ_LEN))  # next-token targets
    )
    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, sampler=sampler)
    criterion = nn.CrossEntropyLoss()

    throughputs = []
    step = 0

    for epoch in range(20):  # Enough to fill NUM_STEPS
        sampler.set_epoch(epoch)
        for x, y in loader:
            if step >= NUM_STEPS:
                break

            t0 = time.perf_counter()
            x = x.to(local_rank)
            y = y.to(local_rank)

            optimizer.zero_grad()
            logits = model(x)  # [B, S, V]
            loss = criterion(logits.view(-1, VOCAB_SIZE), y.view(-1))
            loss.backward()
            optimizer.step()

            elapsed = time.perf_counter() - t0
            tp = (BATCH_SIZE * world_size) / elapsed
            throughputs.append(tp)

            if rank == 0 and step % 5 == 0:
                mem_0 = torch.cuda.memory_allocated(0) / 1e9
                mem_1 = torch.cuda.memory_allocated(1) / 1e9
                print(f"[Step {step:3d}] loss={loss.item():.4f} | "
                      f"{tp:.0f} samples/s | "
                      f"GPU0={mem_0:.2f}GB GPU1={mem_1:.2f}GB")
            step += 1
        if step >= NUM_STEPS:
            break

    if rank == 0:
        avg_tp = sum(throughputs) / len(throughputs)
        peak_0 = torch.cuda.max_memory_allocated(0) / 1e9
        peak_1 = torch.cuda.max_memory_allocated(1) / 1e9
        print(f"\nFINAL SUMMARY [{args.strategy}]")
        print(f"  Avg throughput: {avg_tp:.0f} samples/s")
        print(f"  Peak GPU 0 mem: {peak_0:.2f} GB")
        print(f"  Peak GPU 1 mem: {peak_1:.2f} GB")

    dist.destroy_process_group()


if __name__ == "__main__":
    main()

In [ ]:
print("=" * 60)
print("Strategy 1: NO_SHARD (equivalent to DDP)")
print("=" * 60)
!torchrun --nproc_per_node=2 /kaggle/working/fsdp_compare.py --strategy NO_SHARD

In [ ]:
print("=" * 60)
print("Strategy 2: SHARD_GRAD_OP (ZeRO-2)")
print("=" * 60)
!torchrun --nproc_per_node=2 /kaggle/working/fsdp_compare.py --strategy SHARD_GRAD_OP

In [ ]:
print("=" * 60)
print("Strategy 3: FULL_SHARD (ZeRO-3)")
print("=" * 60)
!torchrun --nproc_per_node=2 /kaggle/working/fsdp_compare.py --strategy FULL_SHARD

**Fill in this table with your results:**

| Strategy | GPU 0 Peak (GB) | GPU 1 Peak (GB) | Throughput (samples/s) |
|----------|-----------------|-----------------|------------------------|
| NO_SHARD | | | |
| SHARD_GRAD_OP | | | |
| FULL_SHARD | | | |

You should see peak memory decrease as you go from NO_SHARD → FULL_SHARD,
and throughput decrease slightly (more communication overhead).

## Part 2: CPU Offloading — Train a Model That Would OOM

CPU offload moves parameters to RAM when not in use. This lets you train models
much larger than your GPU VRAM.

In [ ]:
print("=" * 60)
print("FULL_SHARD + CPU Offload")
print("=" * 60)
!torchrun --nproc_per_node=2 /kaggle/working/fsdp_compare.py --strategy FULL_SHARD --cpu_offload

## Part 3: Saving and Loading FSDP Checkpoints

FSDP checkpointing has a gotcha: the model is sharded across GPUs,
so you can't just call `torch.save(model.state_dict())`.

In [ ]:
%%writefile /kaggle/working/fsdp_checkpoint.py
"""
Demonstrates correct FSDP checkpoint save/load.

Key: use FSDP.state_dict_type() context manager to control how
state is gathered before saving.
"""
import os
import functools
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP, ShardingStrategy, StateDictType
from torch.distributed.fsdp.wrap import transformer_auto_wrap_policy
from torch.distributed.fsdp.fully_sharded_data_parallel import FullStateDictConfig

def main():
    rank = int(os.environ["RANK"])
    local_rank = int(os.environ["LOCAL_RANK"])
    dist.init_process_group(backend="nccl")
    torch.cuda.set_device(local_rank)

    model = nn.Sequential(
        nn.Embedding(1000, 256),
        nn.TransformerEncoderLayer(256, 8, batch_first=True),
        nn.Linear(256, 10)
    )

    wrap_policy = functools.partial(
        transformer_auto_wrap_policy,
        transformer_layer_cls={nn.TransformerEncoderLayer}
    )
    model = FSDP(model, auto_wrap_policy=wrap_policy, device_id=local_rank)

    # Train for one step
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    x = torch.randint(0, 1000, (4, 32)).to(local_rank)
    y = torch.randint(0, 10, (4,)).to(local_rank)
    optimizer.zero_grad()
    out = model(x)
    loss = nn.CrossEntropyLoss()(out[:, 0, :], y)
    loss.backward()
    optimizer.step()

    # ── Saving: use FULL_STATE_DICT to gather all shards onto rank 0 ──────────
    save_policy = FullStateDictConfig(offload_to_cpu=True, rank0_only=True)
    with FSDP.state_dict_type(model, StateDictType.FULL_STATE_DICT, save_policy):
        state_dict = model.state_dict()

    if rank == 0:
        torch.save(state_dict, "/kaggle/working/fsdp_checkpoint.pt")
        print("Checkpoint saved to /kaggle/working/fsdp_checkpoint.pt")
        print(f"Keys: {list(state_dict.keys())[:3]}...")

    dist.barrier()  # Wait for rank 0 to finish saving

    # ── Loading: rank 0 loads, then broadcasts ─────────────────────────────────
    with FSDP.state_dict_type(model, StateDictType.FULL_STATE_DICT, save_policy):
        if rank == 0:
            loaded = torch.load("/kaggle/working/fsdp_checkpoint.pt", weights_only=True)
        else:
            loaded = None  # Non-rank-0 processes pass empty dict
        model.load_state_dict(loaded if rank == 0 else {}, strict=False)

    if rank == 0:
        print("Checkpoint loaded successfully.")

    dist.destroy_process_group()

if __name__ == "__main__":
    main()

In [ ]:
!torchrun --nproc_per_node=2 /kaggle/working/fsdp_checkpoint.py

## Part 4: DeepSpeed ZeRO

DeepSpeed is a library by Microsoft that implements ZeRO with more features.
Config is JSON-based. The API wraps your model similar to FSDP.

In [ ]:
import json

# Write DeepSpeed config files for ZeRO stages 1, 2, 3
for stage in [1, 2, 3]:
    config = {
        "train_micro_batch_size_per_gpu": 4,
        "gradient_accumulation_steps": 1,
        "steps_per_print": 5,
        "optimizer": {
            "type": "AdamW",
            "params": {"lr": 1e-4, "weight_decay": 0.01}
        },
        "fp16": {"enabled": True},
        "zero_optimization": {
            "stage": stage,
            "allgather_partitions": True,
            "allgather_bucket_size": 2e8,
            "reduce_scatter": True,
            "reduce_bucket_size": 2e8,
            "overlap_comm": True,
            "contiguous_gradients": True,
        }
    }
    # ZeRO-3 needs param gather config
    if stage == 3:
        config["zero_optimization"]["stage3_param_persistence_threshold"] = 1e4
        config["zero_optimization"]["stage3_max_live_parameters"] = 1e9

    path = f"/kaggle/working/ds_config_stage{stage}.json"
    with open(path, "w") as f:
        json.dump(config, f, indent=2)
    print(f"Written: {path}")

In [ ]:
%%writefile /kaggle/working/deepspeed_train.py
"""
DeepSpeed ZeRO training script.

Usage:
  deepspeed --num_gpus=2 deepspeed_train.py --stage 2
  deepspeed --num_gpus=2 deepspeed_train.py --stage 3
"""
import os
import argparse
import time
import torch
import torch.nn as nn
import deepspeed
from torch.utils.data import DataLoader, TensorDataset

VOCAB_SIZE = 50000
D_MODEL = 1024
NHEAD = 16
NUM_LAYERS = 12
BATCH_SIZE = 4
SEQ_LEN = 128
NUM_STEPS = 20

class TransformerLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(VOCAB_SIZE, D_MODEL)
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(D_MODEL, NHEAD, D_MODEL*4, dropout=0.1, batch_first=True)
            for _ in range(NUM_LAYERS)
        ])
        self.head = nn.Linear(D_MODEL, VOCAB_SIZE)

    def forward(self, x):
        x = self.embedding(x)
        for layer in self.layers:
            x = layer(x)
        return self.head(x)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--stage", type=int, default=2, choices=[1, 2, 3])
    # DeepSpeed adds its own args; use parse_known_args
    args, _ = parser.parse_known_args()

    local_rank = int(os.environ.get("LOCAL_RANK", 0))

    model = TransformerLM()
    total_params = sum(p.numel() for p in model.parameters())
    if local_rank == 0:
        print(f"Stage: ZeRO-{args.stage} | Params: {total_params:,}")

    ds_config = f"/kaggle/working/ds_config_stage{args.stage}.json"

    # DeepSpeed initialize — handles distributed setup, model wrapping, optimizer
    model_engine, optimizer, _, _ = deepspeed.initialize(
        model=model,
        config=ds_config,
    )

    dataset = TensorDataset(
        torch.randint(0, VOCAB_SIZE, (200, SEQ_LEN)),
        torch.randint(0, VOCAB_SIZE, (200, SEQ_LEN))
    )
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

    step = 0
    throughputs = []

    for epoch in range(100):
        for x, y in loader:
            if step >= NUM_STEPS:
                break

            t0 = time.perf_counter()
            x = x.to(model_engine.local_rank)
            y = y.to(model_engine.local_rank)

            logits = model_engine(x)
            loss = nn.CrossEntropyLoss()(logits.view(-1, VOCAB_SIZE), y.view(-1))

            model_engine.backward(loss)  # replaces loss.backward()
            model_engine.step()          # replaces optimizer.step()

            elapsed = time.perf_counter() - t0
            tp = BATCH_SIZE / elapsed
            throughputs.append(tp)

            if local_rank == 0 and step % 5 == 0:
                mem = torch.cuda.memory_allocated(local_rank) / 1e9
                print(f"[Step {step:3d}] loss={loss.item():.4f} | {tp:.0f}/s | mem={mem:.2f}GB")
            step += 1
        if step >= NUM_STEPS:
            break

    if local_rank == 0:
        avg_tp = sum(throughputs) / len(throughputs)
        peak = torch.cuda.max_memory_allocated(local_rank) / 1e9
        print(f"\nDeepSpeed ZeRO-{args.stage}: {avg_tp:.0f} samples/s | peak mem: {peak:.2f} GB")


if __name__ == "__main__":
    main()

In [ ]:
print("=" * 60)
print("DeepSpeed ZeRO-2")
print("=" * 60)
!deepspeed --num_gpus=2 /kaggle/working/deepspeed_train.py --stage 2

In [ ]:
print("=" * 60)
print("DeepSpeed ZeRO-3")
print("=" * 60)
!deepspeed --num_gpus=2 /kaggle/working/deepspeed_train.py --stage 3

## Checkpoint Questions

1. What specifically does ZeRO-2 shard that ZeRO-1 does not?
2. If you have 2 GPUs and a model with 1B parameters, how much memory does each GPU use for optimizer state under ZeRO-3 vs DDP? (Assume Adam optimizer, fp32 optimizer states, fp16 params)
3. Why can't you use `torch.save(fsdp_model.state_dict())` the same way as a regular model?
4. What additional communication operations does ZeRO-3 require that DDP does not?

**Answers:**
1. ZeRO-2 shards gradients in addition to optimizer states. ZeRO-1 only shards optimizer states.
2. DDP: 8 bytes/param × 1B = 8 GB optimizer state per GPU. ZeRO-3: 8/2 = 4 GB per GPU.
3. With FSDP, the model's parameters are split across GPUs. Calling `state_dict()` on one rank only returns that rank's shard. You need `FULL_STATE_DICT` mode to gather all shards.
4. ZeRO-3 requires AllGather before each layer's forward (to reconstruct full params), and ReduceScatter after backward (to shard gradients). DDP only requires AllReduce after backward.

---

## Next: Open `../05_accelerate/notebook.ipynb`